# Notebook 4c: Baseline 3, DANN (domain-adversarial training)

**The idea.** Train a classifier so that its internal features **cannot reveal which domain an image came from**. A
second small network, the *domain discriminator*, tries to tell lab features from field features. The feature layer is
trained to *fool* it, while still classifying lab images correctly. If the discriminator can no longer tell the domains
apart, the features are "domain-invariant" (Ganin et al., 2016).

**The trick that makes it work: the gradient reversal layer (GRL).** It does nothing when data flows forward. When
the learning signal flows backward, it **flips its sign**. So the discriminator learns to detect the domain, while the
feature layer, receiving the flipped signal, learns to *hide* it. Two opposing goals in one network.

### How the paper did it (Section 2.4)
"A feature-space DANN-style baseline on top of fixed penultimate ResNet-50 features. A small bottleneck classifier was
trained with source-label supervision on PlantVillage features, while a gradient reversal branch fed a binary domain
discriminator trained to distinguish PlantVillage from unlabeled PlantDoc calibration features." They used it as a
**negative-result control**: in their results it made accuracy *worse* (0.252, below the 0.321 of the unadapted model)
and very unstable across splits.

### What the paper does not say (our choices)
Layer sizes, learning rate, number of epochs, how training is stopped, and how the features are scaled. We choose
sensible defaults, listed in Step 1 and explained where they are used. Expect the same *kind* of behaviour, not
identical numbers.

### What we use
* **Source:** the PlantVillage **training** features (notebook 02b).
* **Target:** the ~200 **calibration** images of the split, **without labels**.
* Everything works on the 2048-number features, so no images are needed and a CPU is enough (a GPU is faster).

### Plan
1. Settings
2. Load the features
3. The gradient reversal layer and the model
4. Training on one split
5. All 10 splits, then save
6. Evaluate with notebook 3b

## Step 1: Settings

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

In [4]:
# ---- Edit these -------------------------------------------------------------
QUICK_TEST = False                 # True = 2 splits and 2 epochs, saved under a _QUICK name. False = the real run.

METHOD = "dann"
SOURCE_METHOD = "source_only"
TRAIN_SEED = 42

OUT_DIR = Path("outputs")
MANIFEST = OUT_DIR / "master_manifest.csv"
SEEDS = list(range(42, 52))       # the 10 Setting B splits

# Our choices (the paper does not specify them)
BOTTLENECK = 256                  # size of the bottleneck layer
BATCH_SIZE = 128                  # source images per step (and the same number of target images)
EPOCHS = 20                       # passes over the source features
PATIENCE = 5                      # stop if source-validation accuracy has not improved for this many epochs
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.01
EPS = 1e-6
# -----------------------------------------------------------------------------

if QUICK_TEST:
    EPOCHS = 2
seeds_to_run = SEEDS[:2] if QUICK_TEST else SEEDS
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device, "| splits to run:", seeds_to_run, "| epochs:", EPOCHS)
if QUICK_TEST:
    print("*** QUICK TEST: 2 splits and 2 epochs. Set QUICK_TEST = False for the real run. ***")

Using device: cuda | splits to run: [42, 43, 44, 45, 46, 47, 48, 49, 50, 51] | epochs: 20


## Step 2: Load the features

* From notebook 02: PlantDoc features (known and unknown), the PlantVillage validation features and labels.
* From notebook 02b: the PlantVillage **training** features and labels.

We also **standardise** all features with the PlantVillage training statistics: subtract the training average and divide
by the training spread, feature by feature. Small neural networks train much more smoothly on standardised inputs. We use
the *source* statistics for both domains, so this step does not adapt anything to the target.

In [5]:
source = np.load(OUT_DIR / f"outputs_{SOURCE_METHOD}_seed{TRAIN_SEED}.npz", allow_pickle=True)
classes = list(source["classes"])
n_classes = len(classes)

train = np.load(OUT_DIR / f"features_pv_train_{SOURCE_METHOD}_seed{TRAIN_SEED}.npz", allow_pickle=True)
train_features = train["features"].astype(np.float32)
train_labels = train["labels"]

val_features = source["pv_val_features"].astype(np.float32)
val_labels = source["pv_val_labels"]
known_features = source["pd_known_features"].astype(np.float32)
unknown_features = source["pd_unknown_features"].astype(np.float32)
field_labels = source["pd_known_labels"]
known_paths = source["pd_known_paths"]

print("source train:", train_features.shape, "| source validation:", val_features.shape)
print("field known:", known_features.shape, "| field unknown:", unknown_features.shape)

source train: (18601, 2048) | source validation: (3986, 2048)
field known: (2018, 2048) | field unknown: (466, 2048)


In [6]:
mean, std = train_features.mean(axis=0), train_features.std(axis=0)

def standardise(x):
    return (x - mean) / (std + EPS)

# Move everything to the GPU/CPU as tensors, standardised
to_tensor = lambda x: torch.from_numpy(standardise(x)).to(device)
src_x, val_x = to_tensor(train_features), to_tensor(val_features)
known_x, unknown_x = to_tensor(known_features), to_tensor(unknown_features)
src_y = torch.from_numpy(train_labels).long().to(device)
val_y = torch.from_numpy(val_labels).long().to(device)

# Which known image is calibration / test in each split
manifest = pd.read_csv(MANIFEST, keep_default_na=False)
split_columns = [f"B_seed{s}" for s in SEEDS]
known_rows = manifest[(manifest["dataset"] == "pd") & (manifest["role"] == "pd_known")].set_index("path")
split_info = known_rows.loc[known_paths, split_columns].reset_index(drop=True)

## Step 3: The gradient reversal layer and the model

**Gradient reversal.** Forward: returns its input unchanged. Backward: multiplies the incoming learning signal by
**-lambda**. The number lambda controls how strongly the feature layer is pushed to hide the domain. We start it at 0
and raise it gradually during training (a standard schedule), so the model first learns to classify and only later
learns to blur the domains.

**The model** has three parts:
* a **bottleneck** that squeezes the 2048 features down to 256;
* a **classifier** that predicts the 21 diseases from the bottleneck (trained with lab labels);
* a **domain discriminator** that predicts "lab or field?" from the bottleneck, through the gradient reversal layer.

In [7]:
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)                      # forward: change nothing

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None    # backward: flip the sign of the learning signal


class DANN(nn.Module):
    def __init__(self, in_dim, n_classes, bottleneck=BOTTLENECK):
        super().__init__()
        self.bottleneck = nn.Sequential(nn.Linear(in_dim, bottleneck), nn.ReLU())
        self.classifier = nn.Linear(bottleneck, n_classes)
        self.discriminator = nn.Sequential(nn.Linear(bottleneck, 256), nn.ReLU(), nn.Linear(256, 1))

    def forward(self, x, lambd=0.0):
        feat = self.bottleneck(x)
        return self.classifier(feat), self.discriminator(GradReverse.apply(feat, lambd)), feat

## Step 4: Training on one split

Each step uses a batch of **source** features (with labels) and a batch of **target** features (without labels; they
are drawn from the split's ~200 calibration images, over and over, because there are so few). The loss has two parts:

* **classification loss** on the source batch (cross-entropy with the true labels);
* **domain loss** on source *and* target batches (binary cross-entropy: source = 0, target = 1), which, through the
  gradient reversal layer, pushes the bottleneck to make the two domains indistinguishable.

**Stopping.** We have no target labels, so we watch accuracy on the PlantVillage **validation** features and keep
the best epoch. (This says nothing about how well the model adapts, which is part of why adversarial training is hard to
tune without target labels.)

In [8]:
def train_dann(target_x, seed, verbose=False):
    torch.manual_seed(seed)
    net = DANN(src_x.shape[1], n_classes).to(device)
    optimizer = torch.optim.AdamW(net.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    steps_per_epoch = len(src_x) // BATCH_SIZE
    total_steps = EPOCHS * steps_per_epoch
    best_acc, best_state, bad_epochs = -1.0, None, 0

    for epoch in range(EPOCHS):
        net.train()
        order = torch.randperm(len(src_x), device=device)

        for step in range(steps_per_epoch):
            progress = (epoch * steps_per_epoch + step) / total_steps
            lambd = 2.0 / (1.0 + np.exp(-10.0 * progress)) - 1.0       # rises smoothly from 0 towards 1

            idx = order[step * BATCH_SIZE:(step + 1) * BATCH_SIZE]
            xs, ys = src_x[idx], src_y[idx]
            xt = target_x[torch.randint(len(target_x), (BATCH_SIZE,), device=device)]   # unlabelled target batch

            class_logits, dom_src, _ = net(xs, lambd)
            _, dom_tgt, _ = net(xt, lambd)

            classification_loss = F.cross_entropy(class_logits, ys)
            domain_logits = torch.cat([dom_src, dom_tgt]).squeeze(1)
            domain_labels = torch.cat([torch.zeros(len(xs)), torch.ones(len(xt))]).to(device)
            domain_loss = F.binary_cross_entropy_with_logits(domain_logits, domain_labels)

            optimizer.zero_grad()
            (classification_loss + domain_loss).backward()
            optimizer.step()

        # check on the source validation features
        net.eval()
        with torch.no_grad():
            val_acc = float((net(val_x)[0].argmax(dim=1) == val_y).float().mean())
        if verbose:
            print(f"epoch {epoch + 1:2d} | class loss {classification_loss.item():.3f} | "
                  f"domain loss {domain_loss.item():.3f} | lambda {lambd:.2f} | source val accuracy {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc, bad_epochs = val_acc, 0
            best_state = {k: v.clone() for k, v in net.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                break

    net.load_state_dict(best_state)
    return net.eval()


@torch.no_grad()
def logits_of(net, x):
    return net(x)[0].cpu().numpy().astype(np.float32)

In [9]:
seed = 42
is_cal = (split_info[f"B_seed{seed}"] == "cal").to_numpy()
is_test = ~is_cal

target_x = known_x[torch.from_numpy(is_cal).to(device)]       # this split's calibration images, NO labels
print(f"Split {seed}: {len(target_x)} unlabelled target images")

start = time.time()
net = train_dann(target_x, seed, verbose=True)
print(f"Trained in {time.time() - start:.0f} s")

C:\Users\MIKE\AppData\Local\Temp\ipykernel_21152\3625505851.py:5: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  target_x = known_x[torch.from_numpy(is_cal).to(device)]       # this split's calibration images, NO labels


Split 42: 202 unlabelled target images
epoch  1 | class loss 0.002 | domain loss 0.373 | lambda 0.24 | source val accuracy 0.9960
epoch  2 | class loss 0.023 | domain loss 0.302 | lambda 0.46 | source val accuracy 0.9892
epoch  3 | class loss 0.002 | domain loss 0.240 | lambda 0.63 | source val accuracy 0.9920
epoch  4 | class loss 0.000 | domain loss 0.155 | lambda 0.76 | source val accuracy 0.9952
epoch  5 | class loss 0.000 | domain loss 0.051 | lambda 0.85 | source val accuracy 0.9960
epoch  6 | class loss 0.000 | domain loss 0.151 | lambda 0.90 | source val accuracy 0.9955
Trained in 6 s


In [10]:
new_logits = logits_of(net, known_x)
old_logits = source["pd_known_logits"]                         # the original fine-tuned network

def accuracy(logits, mask):
    return float((logits[mask].argmax(axis=1) == field_labels[mask]).mean())

print(f"Accuracy on the {is_test.sum()} test images (labels used only for scoring):")
print(f"  original network: {accuracy(old_logits, is_test):.4f}")
print(f"  DANN:             {accuracy(new_logits, is_test):.4f}")

Accuracy on the 1816 test images (labels used only for scoring):
  original network: 0.2390
  DANN:             0.2307


**Questions to think about:**
* The paper found DANN **below** the original model (about 0.252 vs 0.321) with large variation between splits.
  Do you see something similar?
* Look at the *domain loss* across epochs. Roughly 0.69 means the discriminator is guessing (the domains look alike). A
  value near 0 means it separates them easily. What does that suggest about how well the "hiding" worked?
* With only ~200 target images, what could the discriminator do instead of learning real domain differences?

## Step 5: All splits, then save

A fresh model is trained for each split, using that split's calibration images as the unlabelled target set. We keep
the logits for all known and unknown PlantDoc images.

In [11]:
save_dict = {"classes": np.array(classes)}
for key in ["epochs_run", "debug_mode"]:
    if key in source.files:
        save_dict[key] = source[key]
for g in ["pv_val", "pv_test"]:
    save_dict[f"{g}_logits"] = source[f"{g}_logits"]
for g in ["pv_val", "pv_test", "pd_known", "pd_unknown"]:
    save_dict[f"{g}_labels"] = source[f"{g}_labels"]
    save_dict[f"{g}_paths"] = source[f"{g}_paths"]

for seed in seeds_to_run:
    start = time.time()
    is_cal = (split_info[f"B_seed{seed}"] == "cal").to_numpy()
    net = train_dann(known_x[torch.from_numpy(is_cal).to(device)], seed)

    save_dict[f"pd_known_logits_B_seed{seed}"] = logits_of(net, known_x)
    save_dict[f"pd_unknown_logits_B_seed{seed}"] = logits_of(net, unknown_x)
    print(f"split {seed}: {time.time() - start:.0f} s | test accuracy "
          f"{accuracy(save_dict[f'pd_known_logits_B_seed{seed}'], ~is_cal):.4f}")

suffix = "_QUICK" if QUICK_TEST else ""
output_file = OUT_DIR / f"outputs_{METHOD}_seed{TRAIN_SEED}{suffix}.npz"
np.savez_compressed(output_file, **save_dict)
print("Saved", output_file)

split 42: 3 s | test accuracy 0.2307
split 43: 4 s | test accuracy 0.2181
split 44: 4 s | test accuracy 0.2148
split 45: 7 s | test accuracy 0.1988
split 46: 6 s | test accuracy 0.2006
split 47: 9 s | test accuracy 0.2137
split 48: 5 s | test accuracy 0.2138
split 49: 8 s | test accuracy 0.1977
split 50: 4 s | test accuracy 0.2043
split 51: 5 s | test accuracy 0.2048
Saved outputs\outputs_dann_seed42.npz


## Step 6: Evaluate with notebook 3b

Run once with `QUICK_TEST = True` to check that everything works, then set it to `False`, restart the kernel and run
again. The real file is `outputs_dann_seed0.npz`. Then add `"dann"` to `METHODS` in notebook 3b.

**The paper's reference values for DANN** (Table 10): accuracy 0.252, calibrated ECE 0.245, calibrated Brier 0.975,
unknown AUROC 0.577, AUPR 0.251, Risk@80 0.717, Risk@50 0.645. Their accuracy 95% interval was [0.227, 0.277], so
expect a wide spread across splits.

### Next
With the three baselines done, we compare all of them with the paper's Table 10. After that come the research methods:
**Deep CORAL and JMMD** (deep feature alignment), then **JDOT and DeepJDOT** (optimal transport).